In [2]:
### IMPORTS
!pip install langchain langchain-ollama langchain-community itext2kg pymupdf neo4j




INFO: pip is looking at multiple versions of itext2kg to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of itext2kg to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide th

ModuleNotFoundError: No module named 'numpy.char'

In [20]:
import requests
from itext2kg import iText2KG_Star, DocumentsDistiller
import gc
import yaml
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field, field_validator
import time
import warnings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from difflib import SequenceMatcher
import os
import re
import asyncio
from typing import List, Optional
from itext2kg.logging_config import get_logger
from langchain_core.caches import BaseCache
import fitz
from neo4j import GraphDatabase

warnings.filterwarnings("ignore", category=UserWarning, module="pydantic._internal._generate_schema")
ChatOpenAI.BaseCache = BaseCache
ChatOpenAI.model_rebuild()
ChatOllama.BaseCache = BaseCache
ChatOllama.model_rebuild()

In [3]:
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [94]:
import subprocess, time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(8)  # give it more time
print("Ollama server started")

Ollama server started


In [76]:
!ollama pull finki_ukim/vezilka:4b-it-q4_K_M
print("Model pulled successfully")


Model pulled successfully


In [65]:
!pip install sentence-transformers langchain-huggingface -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 15.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.7 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.
langchain 0.3.30 requires langchain-core<1.0.0,>=0.3.85, but you have langchain-core 1.4.1 which is incompatible.
itext2kg 1.0.0 requires langchain-core<0.4.0,>=0.3.69, but you have langchain-core 1.4.1 which is incompatible.
langchain-openai 0.2.14 requires langchain-core<0.4.0,>=0.3.27, but you have langchain-core 1.4.1 which is incompatible.


In [67]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings_test = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

test = embeddings_test.embed_query("Географија е наука за Земјата.")
print(f"✅ Embeddings working, vector size: {len(test)}")

/tmp/ipykernel_2885/3711823623.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_test = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

✅ Embeddings working, vector size: 1024


In [57]:
!ollama list

r = requests.get("http://localhost:11434/api/tags")
print(r.json())

NAME                             ID              SIZE      MODIFIED               
finki_ukim/vezilka:4b-it-fp16    b06e7a64db53    7.8 GB    Less than a second ago    
{'models': [{'name': 'finki_ukim/vezilka:4b-it-fp16', 'model': 'finki_ukim/vezilka:4b-it-fp16', 'modified_at': '2026-06-06T11:42:17.225041751Z', 'size': 7767804219, 'digest': 'b06e7a64db534aa6f2745c8224fed99722a7bcac93fc665e867f4a3c887af154', 'details': {'parent_model': '/root/.ollama/models/blobs/sha256-29db5e2b3bad61cc450b19c4dd0631ac0aedadd9405740f608a2a7ed4f371cf2', 'format': 'gguf', 'family': 'gemma3', 'families': ['gemma3'], 'parameter_size': '3.9B', 'quantization_level': 'F16', 'context_length': 131072, 'embedding_length': 2560}, 'capabilities': ['completion']}]}


In [12]:
!nvidia-smi


Sat Jun  6 11:16:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P0             25W /   70W |    8083MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
import requests

def test_ollama_model(model_name: str, prompt: str = "Кажи ми нешто на македонски."):
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": model_name,
                "prompt": prompt,
                "stream": False
            },
            timeout=300
        )
        if response.status_code == 200:
            result = response.json()
            print("✅ Model is working!")
            print(f"Prompt: {prompt}")
            print(f"Response: {result['response']}")
            return True
        else:
            print(f"❌ Model returned status {response.status_code}: {response.text}")
            return False
    except Exception as e:
        print(f"❌ Error testing model: {e}")
        return False

test_ollama_model("finki_ukim/vezilka:4b-it-fp16")

✅ Model is working!
Prompt: Кажи ми нешто на македонски.
Response: Секако! Јас сум помошник за вештачка интелигенција и еве неколку информации што би можеле да ги сметате за интересни:

1. Може ли да ви помогнам со нешто? 
2. Ако сакате, можете да побарате македонски текст или превод на англиски јазик.


True

In [16]:
### PDF READER
from typing import Optional
from google.colab import files
import fitz

def upload_and_read_pdf(output_path: str = "text.txt", start_page: int = 0, end_page: Optional[int] = None):
    """Upload a PDF from local machine and extract text."""
    print("Select a PDF file to upload...")
    uploaded = files.upload()

    if not uploaded:
        print("❌ No file uploaded")
        return "", ""

    pdf_filename = list(uploaded.keys())[0]
    print(f"✅ Uploaded: {pdf_filename}")

    return pdf_filename, read_pdf_to_text(pdf_filename, output_path, start_page, end_page)

def read_pdf_to_text(pdf_path: str, output_path: str = "text.txt", start_page: int = 0, end_page: Optional[int] = None):
    """Extract text from a PDF and save it as UTF-8 text."""
    try:
        text = ""
        doc = fitz.open(pdf_path)
        total_pages = len(doc)
        if end_page is None or end_page > total_pages:
            end_page = total_pages

        for page_num in range(start_page, end_page):
            page = doc.load_page(page_num)
            page_text = page.get_text()
            text += f"\n\n[PAGE {page_num + 1}]\n{page_text}"

        with open(output_path, "w", encoding="utf-8") as f:
            f.write(text)

        print(f"✅ Extracted {len(text)} characters from pages {start_page + 1}-{end_page} of {total_pages}")
        print(f"Saved to: {output_path}")
        return text
    except Exception as e:
        print(f"❌ Error reading PDF: {e}")
        return ""

# Upload and read — adjust page range as needed
pdf_path, extracted_text = upload_and_read_pdf(
    output_path="text.txt",
    start_page=0,
    end_page=20       # small range first to test, set to None for full book
)

Select a PDF file to upload...


Saving Geografija I godina gimnazija_MKD.pdf to Geografija I godina gimnazija_MKD.pdf
✅ Uploaded: Geografija I godina gimnazija_MKD.pdf
✅ Extracted 40394 characters from pages 1-20 of 128
Saved to: text.txt


In [77]:

config = {
    "llm": {
        "model_type": "ollama",
        "model_name": "finki_ukim/vezilka:4b-it-q4_K_M",
        "temperature": 0.1,
    },
    "paths": {
        "output_dir": "./output",
        "triples_file": "textbook_triples_v1.txt",
        "semantic_blocks_file": "textbook_semanticBlocks_v1.txt"
    }
}

os.makedirs(config["paths"]["output_dir"], exist_ok=True)
print("Config loaded")

Config loaded


In [78]:
### LLM AND EMBEDDINGS
def initialize_llm(config):
    if not config:
        return None, None
    llm_config = config.get("llm", {})
    try:
        if llm_config.get("model_type") == "ollama":
            model_name = llm_config.get("model_name", "finki_ukim/vezilka:4b-it-fp16")
            llm = ChatOllama(
                model=model_name,
                temperature=llm_config.get("temperature", 0.1),
                base_url="http://localhost:11434",
                format="json",
                num_predict=1500
            )
            embeddings = HuggingFaceEmbeddings(
                model_name="intfloat/multilingual-e5-large",
                model_kwargs={"device": "cuda"},
                encode_kwargs={"normalize_embeddings": True}
            )
            print(f"✅ LLM initialized: {model_name}")
            print(f"✅ Embeddings initialized: multilingual-e5-large (cuda)")
            return llm, embeddings
        print("Only 'ollama' model type is supported")
        return None, None
    except Exception as e:
        print(f"❌ Error initializing LLM: {e}")
        return None, None

In [84]:
### CUSTOM SCHEMA AND PROMPT FOR MACEDONIAN TEXTBOOKS

class MacedonianTextbookKnowledge(BaseModel):
    subject: Optional[str] = Field(default="", description="Наставен предмет: географија или хемија")
    topic: Optional[str] = Field(default="", description="Главна тема на текстот")
    key_concepts: Optional[str] = Field(default="", description="Клучни поими и концепти, разделени со запирка")
    definitions: Optional[str] = Field(default="", description="Дефиниции на поими: 'поим: дефиниција'")
    classifications: Optional[str] = Field(default="", description="Класификации: 'категорија се дели на: A, B, C'")
    processes: Optional[str] = Field(default="", description="Процеси и настани: 'A предизвикува B'")
    properties: Optional[str] = Field(default="", description="Својства на ентитети: 'X има својство Y'")
    causes_and_effects: Optional[str] = Field(default="", description="Причинско-последични односи: 'X предизвикува Y'")
    geography_entities: Optional[str] = Field(default="", description="Географски ентитети: градови, земји, реки, планини, региони")
    chemistry_entities: Optional[str] = Field(default="", description="Хемиски ентитети: елементи, соединенија, реакции, формули")
    relationships: Optional[str] = Field(default="", description="Односи меѓу ентитети: 'A е дел од B', 'X се наоѓа во Y'")

    @field_validator("*", mode="before")
    @classmethod
    def convert_none_to_empty_string(cls, v):
        if v is None:
            return ""
        if isinstance(v, (list, dict)) and not v:
            return ""
        return v


def get_semantic_distiller_query():
    return """
# ДИРЕКТИВИ:
- Ти си експерт за извлекување ФАКТИ од македонски учебници.
- Извлекувај САМО фактички информации - дефиниции, класификации, процеси, својства, односи.
- НИКОГАШ не извлекувај прашања, задачи, инструкции или активности.
- НИКОГАШ не измислувај информации кои не се во текстот.
- Секое поле мора да содржи конкретни факти во форма: 'ентитет - опис' или 'A е B' или 'X има Y'.
- Ако полето нема релевантни факти во текстот, врати празен стринг "".
- Секогаш враќај string вредности, никогаш null или листи.
- Фокусирај се на односи меѓу ентитети кои може да се претворат во (субјект, предикат, објект).

# ПРИМЕРИ НА ДОБРИ ФАКТИ:
- definitions: "литосфера: надворешниот цврст слој на Земјата составен од тектонски плочи"
- processes: "тектонските плочи се движат и создаваат земјотреси и вулкани"
- geography_entities: "Македонија, Охридско Езеро, Река Вардар, Шар Планина"
- causes_and_effects: "движењето на тектонските плочи предизвикува земјотреси и цунами"
- classifications: "карпи се делат на: магматски, седиментни, метаморфни"

# ПРИМЕРИ НА ЛОШИполиња (НИКОГАШ не враќај вакви):
- questions_tasks: "Кои се главните периоди?" <- ова е прашање, не факт
- definitions: "Прашања и задачи за анализа" <- ова не е дефиниција
"""


In [85]:
### HELPER FUNCTIONS

def safe_get_name(entity, default="unknown"):
    try:
        if entity is None:
            return default
        if hasattr(entity, "name") and entity.name:
            return str(entity.name)
        if isinstance(entity, str):
            return entity
        if isinstance(entity, dict) and "name" in entity:
            return str(entity["name"])
    except Exception:
        pass
    return default

TRIVIAL_OBJECTS = {"yes", "no", "none", "all_the_time", "sometimes_present", "unknown", "да", "не", "нема", "непознато", "непозната", "непознат"}
MACEDONIAN_BAD_WORDS = {"it", "this", "that", "they", "he", "she", "something", "anything", "everything", "nothing", "somewhere", "anywhere", "everyone", "anyone", "тоа", "ова", "овој", "оваа", "тие", "тој", "таа", "нешто", "некој", "некоја", "сите", "никој", "ништо", "таму", "тука"}
FORMULA_PATTERN = re.compile(r"^(?:[A-Z][a-z]?\d*)+(?:[+\-()\[\]·.]*\d*)*$")


def clean_entity(entity: str) -> str:
    if not entity:
        return ""
    entity = re.sub(r"\s+", " ", str(entity).strip())
    entity = re.sub(r"[,;:.!?]+$", "", entity)
    entity = entity.strip("()[]{} ")
    if not entity or entity.lower() in TRIVIAL_OBJECTS:
        return ""
    if FORMULA_PATTERN.fullmatch(entity):
        return entity
    return entity


def enhanced_clean_predicate(predicate: str) -> str:
    if not predicate:
        return "relates_to"
    predicate = str(predicate).lower().strip()
    predicate = re.sub(r"[^\w\sа-шѓќљњѕџјА-ШЃЌЉЊЅЏЈ]", "", predicate)
    predicate = re.sub(r"\s+", "_", predicate)
    mappings = {
        "е": "is", "се": "relates_to", "има": "has", "содржи": "contains",
        "составен_од": "consists_of", "се_состои_од": "consists_of",
        "припаѓа_на": "belongs_to", "е_дел_од": "part_of",
        "се_дели_на": "classified_as", "се_класифицира_на": "classified_as",
        "се_формира_со": "formed_by", "се_создава_со": "formed_by", "настанува_со": "formed_by",
        "предизвикува": "causes", "влијае_на": "affects", "се_мери_во": "measured_in",
        "се_користи_за": "used_for", "се_наоѓа_во": "located_in", "опфаќа": "includes",
        "вклучува": "includes", "изучува": "studies", "дефинира": "defines",
        "пример_за": "example_of", "тип_на": "is_type_of", "вид_на": "is_type_of",
        "симбол_за": "symbol_for", "формула_за": "formula_for"
    }
    if predicate in mappings:
        return mappings[predicate]
    english_mappings = {
        "is_a": "is_type_of", "has_a": "has", "contain": "contains", "contains": "contains",
        "belong": "belongs_to", "belongs_to": "belongs_to", "define": "defines", "defines": "defines",
        "include": "includes", "includes": "includes", "cause": "causes", "causes": "causes",
        "affect": "affects", "affects": "affects", "measure": "measured_in", "located": "located_in",
        "part": "part_of", "formed": "formed_by", "consists": "consists_of", "classified": "classified_as",
        "used": "used_for", "studies": "studies"
    }
    for key, value in english_mappings.items():
        if key in predicate:
            return value
    return predicate if len(predicate) >= 2 else "relates_to"


def validate_triple_semantically(subject: str, predicate: str, obj: str) -> bool:
    invalid_patterns = [
        (r".*", r"relates_to", r"(да|не|yes|no|unknown|none)"),
        (r"(ова|тоа|тие|this|that|it)", r".*", r".*"),
        (r".*", r".*", r"(ова|тоа|тие|this|that|it)")
    ]
    for s_pattern, p_pattern, o_pattern in invalid_patterns:
        if re.match(s_pattern, subject, re.IGNORECASE) and re.match(p_pattern, predicate, re.IGNORECASE) and re.match(o_pattern, obj, re.IGNORECASE):
            return False
    return True


def is_valid_triple(subject: str, predicate: str, obj: str) -> bool:
    if len(subject) < 2 or len(obj) < 2:
        return False
    if subject.lower() == obj.lower():
        return False
    if subject.lower() in MACEDONIAN_BAD_WORDS or obj.lower() in MACEDONIAN_BAD_WORDS:
        return False
    if predicate in ["is", "has", "does", "е", "има"] and len(predicate) < 4:
        return False
    return validate_triple_semantically(subject, predicate, obj)


def deduplicate_semantic_blocks(blocks: List[str], similarity_threshold: float = 0.75) -> List[str]:
    if not blocks:
        return blocks
    unique_blocks = []
    for block in blocks:
        duplicate = False
        block_clean = block.lower().strip()
        for existing_block in unique_blocks:
            if SequenceMatcher(None, block_clean, existing_block.lower().strip()).ratio() > similarity_threshold:
                duplicate = True
                if len(block) > len(existing_block):
                    unique_blocks.remove(existing_block)
                    unique_blocks.append(block)
                break
        if not duplicate:
            unique_blocks.append(block)
    return unique_blocks


def save_triples(triples, filename="extracted_triples.txt"):
    os.makedirs(os.path.dirname(filename) or ".", exist_ok=True)
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"# Knowledge Graph Triples\n# Total: {len(triples)} triples\n")
        f.write("# Format: (Subject) --[Predicate]--> (Object)\n\n")
        for i, (s, p, o) in enumerate(triples, 1):
            f.write(f"{i:3d}. ({s}) --[{p}]--> ({o})\n")
        f.write(f"\n# Extraction completed. {len(triples)} unique triples saved.\n")

class PipelineError(Exception):
    pass

async def robust_distillation(chunk: str, distiller, max_retries: int = 3):
    for attempt in range(max_retries):
        try:
            cleaned_chunk = chunk.replace("{", "[").replace("}", "]")
            return await distiller.distill(documents=[cleaned_chunk], IE_query=get_semantic_distiller_query(), output_data_structure=MacedonianTextbookKnowledge)
        except asyncio.TimeoutError as e:
            print(f"    -> Attempt {attempt + 1} timed out: {e}")
        except Exception as e:
            error_type = type(e).__name__
            error_msg = str(e)[:100] + "..." if len(str(e)) > 100 else str(e)
            print(f"    -> Attempt {attempt + 1} failed ({error_type}): {error_msg}")
            if attempt == max_retries - 1:
                raise PipelineError(f"Distillation failed after {max_retries} attempts: {error_type} - {error_msg}")
        if attempt < max_retries - 1:
            wait_time = 2 ** attempt
            print(f"    -> Retrying in {wait_time} seconds...")
            await asyncio.sleep(wait_time)


def process_chunks_in_batches(chunks, batch_size=5):
    for i in range(0, len(chunks), batch_size):
        yield chunks[i:i + batch_size]


def filter_semantic_blocks(blocks: List[str], min_length: int = 30) -> List[str]:
    filtered_blocks = []
    for block in blocks:
        if not block or len(block.strip()) < min_length:
            continue
        trivial_patterns = [r"^\s*$", r"^\w+:\s*(none|unknown|yes|no|да|не|нема|непознато)\s*$", r"^[^:]+:\s*$"]
        if any(re.search(pat, block, flags=re.IGNORECASE) for pat in trivial_patterns):
            continue
        filtered_blocks.append(block)
    return filtered_blocks


def create_semantic_blocks(distilled_article: MacedonianTextbookKnowledge) -> List[str]:
    semantic_blocks = []
    article_dict = distilled_article.model_dump()
    block_config = {
        "subject": "Наставен предмет",
        "topic": "Тема",
        "key_concepts": "Клучни поими",
        "definitions": "Дефиниции",
        "classifications": "Класификации",
        "processes": "Процеси",
        "properties": "Својства",
        "causes_and_effects": "Причинско-последични односи",
        "geography_entities": "Географски ентитети",
        "chemistry_entities": "Хемиски ентитети",
        "relationships": "Односи",
    }

    bad_patterns = [
        r"прашањ", r"задач", r"активност", r"анализ.*учебник",
        r"одговор.*прашањ", r"^\s*\d+\.\s*(кои|како|што|каде|зошто)",
        r"за правење", r"за оценување", r"за разбирање"
    ]

    for key, value in article_dict.items():
        if not value or not isinstance(value, str) or len(value.strip()) < 20:
            continue
        cleaned_value = value.replace("{", "[").replace("}", "]").strip()

        # Skip if it looks like questions/tasks
        if any(re.search(pat, cleaned_value, re.IGNORECASE) for pat in bad_patterns):
            continue

        semantic_blocks.append(f"{block_config.get(key, key)}: {cleaned_value}")
    return semantic_blocks

class ETACalculator:
    def __init__(self, total_items: int):
        self.total_items = max(total_items, 1)
        self.start_time = time.time()
    def get_eta_string(self, current_item: int) -> str:
        elapsed = time.time() - self.start_time
        avg = elapsed / current_item if current_item else 0
        remaining = avg * (self.total_items - current_item)
        progress = (current_item / self.total_items) * 100
        return f"Progress: {progress:.1f}% | Elapsed: {elapsed/60:.1f}min | ETA: {remaining/60:.1f}min"

In [90]:
async def main(input_source: str = "text.txt"):
    print("Starting Macedonian Textbook KG Extraction Pipeline\n")
    if os.path.exists(input_source):
        with open(input_source, "r", encoding="utf-8") as f:
            text = f.read().strip()
    else:
        text = input_source.strip()
    if len(text) < 20:
        print("Text too short")
        return []

    llm, embeddings = initialize_llm(config)
    if llm is None or embeddings is None:
        print("LLM or embeddings failed to initialize")
        return []
    print(f"LLM initialized: {config['llm']['model_name']}")

    output_dir = config.get("paths", {}).get("output_dir", "./Triples and Semantic Blocks")
    triples_file = config.get("paths", {}).get("triples_file", "textbook_triples_v1.txt")
    semantic_blocks_file = config.get("paths", {}).get("semantic_blocks_file", "textbook_semanticBlocks_v1.txt")
    os.makedirs(output_dir, exist_ok=True)

    all_triples = []
    start_time = time.time()
    print("\n=== STEP 1: Document Distillation ===")
    sentences = re.split(r"(?<=[.!?؟])\s+|\n+", text)
    chunks, current_chunk = [], ""
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        if len(current_chunk + sentence) <= 1200:
            current_chunk += sentence + " "
        else:
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
            current_chunk = sentence + " "
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    print(f"Text chunked: {len(text)} chars in {len(chunks)} chunks")

    eta_distiller = ETACalculator(len(chunks))
    try:
        from itext2kg.llm_output_parsing.langchain_output_parser import (
            LangchainOutputParser, ProviderType, PROVIDER_CONFIGS, ProviderConfig
        )
        PROVIDER_CONFIGS[ProviderType.UNKNOWN] = ProviderConfig(
            name="Ollama",
            max_elements_per_batch=1,
            max_tokens_per_batch=4000,
            max_context_window=131072,
            max_pending_requests=None,
            sleep_between_batches=0.5,
        )
        original_detect = LangchainOutputParser._detect_provider
        def patched_detect(self):
            cls = self.model.__class__.__name__.lower()
            mod = self.model.__class__.__module__.lower()
            if "ollama" in cls or "ollama" in mod:
                return ProviderType.UNKNOWN
            return original_detect(self)
        LangchainOutputParser._detect_provider = patched_detect
        print("✅ Ollama provider patch applied")

        distiller = DocumentsDistiller(llm_model=llm)

        import types
        async def patched_extract(self, output_data_structure, contexts, system_query=None):
            if system_query is None:
                system_query = "Act like an experienced information extractor. If you do not find the right information, keep its place empty."
            structured_llm = self.model.with_structured_output(output_data_structure)
            outputs = []
            for i, context in enumerate(contexts):
                prompt = f"# Context: {context}\n\n# Question: {system_query}\n\nAnswer: "
                for attempt in range(3):
                    try:
                        result = await structured_llm.ainvoke(prompt)
                        outputs.append(result)
                        print(f"    ✅ Context {i+1}/{len(contexts)} extracted")
                        break
                    except Exception as e:
                        if attempt == 2:
                            print(f"    ❌ Context {i+1} failed: {str(e)[:80]}")
                            outputs.append(output_data_structure())
                        else:
                            print(f"    ⚠️ Attempt {attempt+1} failed, retrying: {str(e)[:60]}")
                            await asyncio.sleep(2 ** attempt)
                await asyncio.sleep(0.5)
            return outputs

        parser = distiller.langchain_output_parser
        parser.extract_information_as_json_for_context = types.MethodType(patched_extract, parser)
        print("✅ Distiller instance patched")

    except ImportError as e:
        print(f"Failed to import DocumentsDistiller: {e}")
        return []

    all_semantic_blocks = []
    successful_chunks = 0
    BATCH_SIZE = 5
    for batch_num, chunk_batch in enumerate(process_chunks_in_batches(chunks, BATCH_SIZE)):
        print(f"\nSTARTING DISTILLATION BATCH {batch_num + 1}/{(len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE}")
        batch_blocks = []
        for i, chunk in enumerate(chunk_batch):
            global_chunk_idx = batch_num * BATCH_SIZE + i
            print(f"Distilling chunk {global_chunk_idx + 1}/{len(chunks)}...")
            try:
                distilled_article = await robust_distillation(chunk, distiller)
                if distilled_article:
                    chunk_blocks = create_semantic_blocks(distilled_article)
                    batch_blocks.extend(chunk_blocks)
                    successful_chunks += 1
                    print(f"Generated {len(chunk_blocks)} semantic blocks")
                elif len(chunk.strip()) > 50:
                    batch_blocks.append(f"Содржински блок: {chunk.strip()[:500]}...")
            except PipelineError as e:
                print(f"Using fallback block: {e}")
                if len(chunk.strip()) > 50:
                    batch_blocks.append(f"Содржински блок: {chunk.strip()[:500]}...")
            print(eta_distiller.get_eta_string(global_chunk_idx + 1))
        all_semantic_blocks.extend(batch_blocks)
        print(f"Added {len(batch_blocks)} blocks; total blocks so far: {len(all_semantic_blocks)}")
        del batch_blocks
        gc.collect()

    print("\nDistillation Summary:")
    print(f"- Total semantic blocks created: {len(all_semantic_blocks)}")
    print(f"- Successful chunks: {successful_chunks}/{len(chunks)}")
    all_semantic_blocks = filter_semantic_blocks(all_semantic_blocks, min_length=30)
    all_semantic_blocks = deduplicate_semantic_blocks(all_semantic_blocks, similarity_threshold=0.75)
    print(f"Remaining semantic blocks after filtering and deduplication: {len(all_semantic_blocks)}")

    print("\n=== STEP 2: STAR Knowledge Graph Extraction ===")
    if not all_semantic_blocks:
        all_semantic_blocks = [f"Текстуален блок: {chunk[:300]}..." for chunk in chunks if len(chunk.strip()) > 50]
    try:
        logger = get_logger(__name__)
        star_model = iText2KG_Star(llm_model=llm, embeddings_model=embeddings)

        # Patch star_model's parser instance too
        star_parser = star_model.langchain_output_parser
        star_parser.extract_information_as_json_for_context = types.MethodType(patched_extract, star_parser)
        print("✅ Star model instance patched")

        batches = [all_semantic_blocks[i:i+10] for i in range(0, len(all_semantic_blocks), 10)]
        eta_star = ETACalculator(len(batches))
        for batch_idx, batch in enumerate(batches):
            print(f"\nPROCESSING STAR BATCH {batch_idx + 1}/{len(batches)}")
            batch_triples = 0
            for block_idx, block in enumerate(batch):
                try:
                    kg_result = await star_model.build_graph(sections=[block], ent_threshold=0.5, rel_threshold=0.4, observation_date="2026-06-05")
                    if hasattr(kg_result, "relationships") and kg_result.relationships:
                        block_triples = 0
                        for relationship in kg_result.relationships:
                            s = clean_entity(safe_get_name(getattr(relationship, "startEntity", None)))
                            o = clean_entity(safe_get_name(getattr(relationship, "endEntity", None)))
                            p = enhanced_clean_predicate(safe_get_name(getattr(relationship, "name", None), "relates_to"))
                            if is_valid_triple(s, p, o):
                                all_triples.append((s, p, o))
                                batch_triples += 1
                                block_triples += 1
                        print(f"Extracted {block_triples} triples from block {block_idx + 1}")
                except Exception as block_error:
                    error_msg = str(block_error)[:100] + "..." if len(str(block_error)) > 100 else str(block_error)
                    print(f"Block {block_idx + 1} failed ({type(block_error).__name__}): {error_msg}")
            print(f"Extracted {batch_triples} valid triples from this batch")
            print(eta_star.get_eta_string(batch_idx + 1))
    except Exception as star_error:
        error_msg = str(star_error)[:200] + "..." if len(str(star_error)) > 200 else str(star_error)
        print(f"STAR failed ({type(star_error).__name__}): {error_msg}")
        return []

    print("\n=== STEP 3: Post-Processing ===")
    seen, unique_triples = set(), []
    skipped_invalid = 0
    for s, p, o in all_triples:
        clean_triple = (clean_entity(s), enhanced_clean_predicate(p), clean_entity(o))
        if not is_valid_triple(*clean_triple):
            skipped_invalid += 1
            continue
        if clean_triple not in seen:
            seen.add(clean_triple)
            unique_triples.append(clean_triple)
    final_triples = unique_triples
    print(f"\nProcessing completed in {(time.time() - start_time)/60:.1f} minutes")
    print(f"Total triples extracted: {len(all_triples)}")
    print(f"Invalid triples skipped: {skipped_invalid}")
    print(f"Final unique valid triples: {len(final_triples)}")

    triples_path = os.path.join(output_dir, triples_file)
    blocks_path = os.path.join(output_dir, semantic_blocks_file)
    save_triples(final_triples, triples_path)
    print(f"Triples saved to: {triples_path}")
    with open(blocks_path, "w", encoding="utf-8") as f:
        f.write(f"# Semantic Blocks Generated\n# Total: {len(all_semantic_blocks)} blocks\n\n")
        for i, block in enumerate(all_semantic_blocks, 1):
            f.write(f"Block {i}:\n{block}\n{'-'*50}\n\n")
    print(f"Semantic blocks saved to: {blocks_path}")
    if final_triples:
        print("\nSample extracted triples:")
        for i, (s, p, o) in enumerate(final_triples[:10], 1):
            print(f"   {i:2d}. ({s}) --[{p}]--> ({o})")
    return final_triples

In [91]:
### QUICK START: EXTRACT A SMALL TEXTBOOK SECTION FIRST

# Chemistry textbook, pages around the first lesson.
# pdf_file = r"C:\Users\User\Downloads\document_pdf.pdf"
# read_pdf_to_text(pdf_file, output_path="text.txt", start_page=6, end_page=16)

# Geography high school textbook, pages around lithosphere/geological evolution.
# pdf_file = r"C:\Users\User\Downloads\Geografija I godina gimnazija_MKD.pdf"
# read_pdf_to_text(pdf_file, output_path="text.txt", start_page=5, end_page=16)

# Geography 7th grade textbook, pages around Europe.
# pdf_file = r"C:\Users\User\Downloads\GEOGRAFIJA ZA 7 ODDELENIE 2025_konecno.pdf"
# read_pdf_to_text(pdf_file, output_path="text.txt", start_page=5, end_page=16)

In [92]:
# import itext2kg
# import inspect
# from itext2kg.llm_output_parsing import langchain_output_parser
# print(inspect.getsource(langchain_output_parser))

In [95]:
### RUN PIPELINE

import nest_asyncio
import asyncio
nest_asyncio.apply()

async def run_textbook_pipeline():
    triples = await main("text.txt")
    print(f"\nPipeline completed! Found {len(triples)} triples")
    return triples

triples = await run_textbook_pipeline()

Starting Macedonian Textbook KG Extraction Pipeline



Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ LLM initialized: finki_ukim/vezilka:4b-it-q4_K_M
✅ Embeddings initialized: multilingual-e5-large (cuda)
LLM initialized: finki_ukim/vezilka:4b-it-q4_K_M

=== STEP 1: Document Distillation ===
Text chunked: 40391 chars in 34 chunks
✅ Ollama provider patch applied
[2026-06-06 12:49:17] [    INFO] [itext2kg.llm_output_parsing.langchain_output_parser] 🔍 Detected LLM Provider: Ollama
[2026-06-06 12:49:17] [    INFO] [itext2kg.llm_output_parsing.langchain_output_parser] 📊 Rate Limiting Config: 1 requests/batch, 4000 tokens/batch
✅ Distiller instance patched

STARTING DISTILLATION BATCH 1/7
Distilling chunk 1/34...
    ✅ Context 1/1 extracted
Generated 3 semantic blocks
Progress: 2.9% | Elapsed: 0.2min | ETA: 8.0min
Distilling chunk 2/34...
    ✅ Context 1/1 extracted
Generated 2 semantic blocks
Progress: 5.9% | Elapsed: 0.3min | ETA: 4.9min
Distilling chunk 3/34...
    ✅ Context 1/1 extracted
Generated 4 semantic blocks
Progress: 8.8% | Elapsed: 0.4min | ETA: 3.9min
Distilling chunk 4/34..

In [96]:
from google.colab import files
files.download("./output/textbook_triples_v1.txt")
files.download("./output/textbook_semanticBlocks_v1.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [58]:
# Test structured output with Ollama directly
from langchain_ollama import ChatOllama
from pydantic import BaseModel

class TestSchema(BaseModel):
    answer: str = ""

llm_test = ChatOllama(
    model="finki_ukim/vezilka:4b-it-fp16",
    base_url="http://localhost:11434",
    format="json",
    num_predict=200
)

structured = llm_test.with_structured_output(TestSchema)
result = await structured.ainvoke("What is geography? Answer in one sentence.")
print(result)

answer='Географијата е проучување на Земјината физичка и човечка средина, вклучувајќи ги нејзините процеси, системи и структури кои го обликуваат животот и интеракциите на луѓето со нив. Тоа се протега од фундаментално разбирање на геолошките сили до анализа на урбаниот развој и планирање.}'


In [59]:
from itext2kg import DocumentsDistiller
from langchain_ollama import ChatOllama, OllamaEmbeddings

llm_temp = ChatOllama(model="finki_ukim/vezilka:4b-it-fp16", base_url="http://localhost:11434", format="json")
embeddings_temp = OllamaEmbeddings(model="finki_ukim/vezilka:4b-it-fp16", base_url="http://localhost:11434")

distiller_temp = DocumentsDistiller(llm_model=llm_temp)
print([attr for attr in dir(distiller_temp) if 'parser' in attr.lower() or 'llm' in attr.lower() or 'output' in attr.lower() or 'lang' in attr.lower()])

[2026-06-06 11:44:09] [    INFO] [itext2kg.llm_output_parsing.langchain_output_parser] 🔍 Detected LLM Provider: Ollama
[2026-06-06 11:44:09] [    INFO] [itext2kg.llm_output_parsing.langchain_output_parser] 📊 Rate Limiting Config: 1 requests/batch, 4000 tokens/batch
['langchain_output_parser']


❌ Ollama is DOWN - need to restart it


In [ ]:
### NEO4J EXPORT FOR MACEDONIAN TEXTBOOK KNOWLEDGE GRAPH

class Neo4jClient:
    def __init__(self, uri="bolt://localhost:7687", user="neo4j", password="12345678"):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
    def close(self):
        self.driver.close()
    def create_triples(self, triples, label_mapping=None, rel_mapping=None):
        label_mapping = label_mapping or {}
        rel_mapping = rel_mapping or {}
        auto_labels = self._auto_detect_labels(triples)
        for entity, label in auto_labels.items():
            label_mapping.setdefault(entity, label)
        for pred in set(pred for _, pred, _ in triples):
            rel_mapping.setdefault(pred, self._clean_predicate_name(pred))
        print(f"Processing {len(triples)} triples with {len(set(label_mapping.values()))} entity types")
        with self.driver.session() as session:
            for subj, pred, obj in triples:
                session.execute_write(lambda tx: self._create_single_triple(tx, subj, pred, obj, label_mapping, rel_mapping))
    def _auto_detect_labels(self, triples):
        label_mapping = {}
        entities = {x for s, _, o in triples for x in (s, o)}
        patterns = {
            "Subject": [r"^географија$", r"^хемија$"],
            "Grade": [r".*vii.*", r".*7.*одделение.*", r".*седмо.*", r".*i.*година.*", r".*прва.*година.*"],
            "Topic": [r".*супстанци.*", r".*литосфера.*", r".*атмосфера.*", r".*хидросфера.*", r".*биосфера.*", r".*европа.*", r".*азија.*"],
            "ChemicalElement": [r"^водород$", r"^кислород$", r"^јаглерод$", r"^азот$", r"^железо$", r"^бакар$", r"^злато$", r"^сребро$", r"^сулфур$", r"^[A-Z][a-z]?$"],
            "ChemicalCompound": [r".*соединение.*", r"^H2O$", r"^CO2$", r"^NaCl$"],
            "Substance": [r".*супстанц.*", r".*елементарн.*", r".*чиста.*"],
            "Mixture": [r".*смес.*", r".*раствор.*", r".*легур.*"],
            "Formula": [r"^(?:[A-Z][a-z]?\d*)+$", r".*формул.*", r".*равенк.*", r".*симбол.*"],
            "Experiment": [r".*експеримент.*", r".*лаборатор.*", r".*прибор.*"],
            "Process": [r".*ерозија.*", r".*акумулација.*", r".*урбанизација.*", r".*еволуција.*", r".*таложење.*", r".*ладење.*", r".*стврднување.*", r".*одделување.*"],
            "GeographicFeature": [r".*планин.*", r".*река.*", r".*езеро.*", r".*море.*", r".*океан.*", r".*полуостров.*", r".*залив.*", r".*проток.*", r".*релјеф.*"],
            "Continent": [r"^европа$", r"^азија$", r"^африка$", r"^австралија$", r".*америка.*"],
            "Region": [r".*европа$", r".*балкан.*", r".*регион.*"],
            "Country": [r"^македонија$", r"^германија$", r"^франција$", r"^украина$", r"^русија$", r"^индија$", r"^кина$", r"^јапонија$", r"^туркија$"],
            "City": [r"^скопје$", r"^минск$", r"^рига$", r"^киев$", r"^москва$", r"^париз$", r"^берлин$"],
            "Unit": [r".*km.*", r".*km2.*", r".*hpa.*", r".*mb.*", r".*милибар.*", r".*степен.*", r".*процент.*"],
            "Property": [r".*својств.*", r".*густина.*", r".*температура.*", r".*притисок.*", r".*состав.*"]
        }
        for entity in entities:
            entity_lower = entity.lower()
            label_mapping[entity] = "Concept"
            for label, pats in patterns.items():
                if any(re.match(pat, entity_lower, re.IGNORECASE) for pat in pats):
                    label_mapping[entity] = label
                    break
        return label_mapping
    def _clean_predicate_name(self, predicate):
        common = {"defines":"DEFINES", "contains":"CONTAINS", "consists_of":"CONSISTS_OF", "classified_as":"CLASSIFIED_AS", "part_of":"PART_OF", "is_type_of":"IS_TYPE_OF", "formed_by":"FORMED_BY", "causes":"CAUSES", "affects":"AFFECTS", "measured_in":"MEASURED_IN", "used_for":"USED_FOR", "located_in":"LOCATED_IN", "includes":"INCLUDES", "studies":"STUDIES", "relates_to":"RELATES_TO", "example_of":"EXAMPLE_OF", "formula_for":"FORMULA_FOR", "symbol_for":"SYMBOL_FOR"}
        if predicate in common:
            return common[predicate]
        cleaned = re.sub(r"[^a-zA-Z0-9_]", "_", predicate.upper())
        cleaned = re.sub(r"_+", "_", cleaned).strip("_")
        return cleaned[:50] if cleaned else "RELATES_TO"
    def _create_single_triple(self, tx, subj, pred, obj, label_mapping, rel_mapping):
        query = f"""
        MERGE (s:{label_mapping.get(subj, 'Concept')} {{name: $subj}})
        MERGE (o:{label_mapping.get(obj, 'Concept')} {{name: $obj}})
        MERGE (s)-[r:{rel_mapping.get(pred, 'RELATES_TO')}]->(o)
        RETURN s, r, o
        """
        tx.run(query, subj=subj, obj=obj)

label_mapping = {
    "Географија": "Subject", "Хемија": "Subject", "Супстанци": "Topic", "Литосфера": "Topic",
    "Атмосфера": "Topic", "Хидросфера": "Topic", "Биосфера": "Topic", "Европа": "Continent",
    "Азија": "Continent", "Земјина кора": "GeographicFeature", "Магматски карпи": "GeographicFeature",
    "Седиментни карпи": "GeographicFeature", "Метаморфни карпи": "GeographicFeature",
    "Експеримент": "Experiment", "Чиста супстанца": "Substance", "Смеса": "Mixture",
    "Елементарна супстанца": "Substance", "Соединение": "ChemicalCompound"
}
rel_mapping = {"defines":"DEFINES", "contains":"CONTAINS", "consists_of":"CONSISTS_OF", "classified_as":"CLASSIFIED_AS", "part_of":"PART_OF", "is_type_of":"IS_TYPE_OF", "formed_by":"FORMED_BY", "causes":"CAUSES", "affects":"AFFECTS", "measured_in":"MEASURED_IN", "used_for":"USED_FOR", "located_in":"LOCATED_IN", "includes":"INCLUDES", "studies":"STUDIES", "relates_to":"RELATES_TO"}

def load_triples_from_file(path="Triples and Semantic Blocks/textbook_triples_v1.txt"):
    triples = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                if line.strip().startswith(tuple("0123456789")):
                    line = line.split(". ", 1)[1] if ". " in line else line
                subj, rest = line.split("--[")
                pred, obj = rest.split("]-->")
                triples.append((subj.strip()[1:-1], pred.strip(), obj.strip()[1:-1]))
            except Exception:
                continue
    return triples

# Example usage after running the pipeline:
# triples = load_triples_from_file("Triples and Semantic Blocks/textbook_triples_v1.txt")
# neo4j_client = Neo4jClient(uri="bolt://localhost:7687", user="neo4j", password="12345678")
# neo4j_client.create_triples(triples, label_mapping, rel_mapping)
# neo4j_client.close()